# BERT (Bidirectional Encoder Representations from Transformers)
This notebook implements BERT with Masked Language Modeling (MLM) task.

## Imports and Dataset

In [1]:
import tensorflow as tf
from tensorflow.keras.layers import (
    TextVectorization,
    Embedding,
    Dense,
    LayerNormalization,
    MultiHeadAttention,
    Masking
)
from tensorflow.keras import Model
import numpy as np

data = [
    "the quick brown fox jumps over the lazy dog",
    "machine learning is a subset of artificial intelligence",
    "transformers have revolutionized natural language processing",
    "attention is all you need for deep learning",
    "bert is a bidirectional encoder representations from transformers",
    "masked language modeling is the primary task for bert",
    "bert performs well on many downstream nlp tasks",
    "pretraining on large corpora improves model performance",
    "fine tuning bert on specific tasks yields great results",
    "contextual word embeddings capture semantic meaning"
]

## Tokenization

In [2]:
vocab_size = 500
sequence_length = 20

text_vectorization = TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length
)

text_vectorization.adapt(data)

tokens = text_vectorization(data)

## Create Masked LM Dataset

In [3]:
def create_masked_lm_data(token_ids, mask_prob=0.15):
    """Create masked language modeling training data"""
    input_ids = token_ids.numpy().copy()
    labels = token_ids.numpy().copy()
    
    mask_token_id = vocab_size - 1
    
    for i in range(input_ids.shape[0]):
        for j in range(input_ids.shape[1]):
            if np.random.random() < mask_prob and input_ids[i, j] != 0:
                input_ids[i, j] = mask_token_id
    
    return tf.constant(input_ids, dtype=tf.int64), tf.constant(labels, dtype=tf.int64)

masked_inputs, mlm_labels = create_masked_lm_data(tokens)

## Positional Embedding Layer

In [4]:
class PositionalEmbedding(tf.keras.layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim):
        super().__init__()

        self.token_embedding = Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim
        )

        self.position_embedding = Embedding(
            input_dim=sequence_length,
            output_dim=embed_dim
        )

    def call(self, inputs):
        length = tf.shape(inputs)[-1]

        positions = tf.range(
            start=0,
            limit=length,
            delta=1
        )

        token_embeddings = self.token_embedding(inputs)
        position_embeddings = self.position_embedding(positions)

        return token_embeddings + position_embeddings

## Transformer Encoder Layer

In [5]:
class TransformerEncoderLayer(tf.keras.layers.Layer):

    def __init__(self, embed_dim, dense_dim, num_heads):
        super().__init__()

        self.attention = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        self.ffn = tf.keras.Sequential([
            Dense(dense_dim, activation="relu"),
            Dense(embed_dim)
        ])

        self.layernorm1 = LayerNormalization()
        self.layernorm2 = LayerNormalization()

    def call(self, inputs):
        attention_output = self.attention(
            query=inputs,
            value=inputs,
            key=inputs
        )

        out1 = self.layernorm1(
            inputs + attention_output
        )

        ffn_output = self.ffn(out1)

        return self.layernorm2(
            out1 + ffn_output
        )

## Build BERT Model

In [6]:
embed_dim = 128
dense_dim = 256
num_heads = 4
num_layers = 2

encoder_input = tf.keras.Input(
    shape=(sequence_length,),
    dtype="int64"
)

embeddings = PositionalEmbedding(
    sequence_length,
    vocab_size,
    embed_dim
)(encoder_input)

encoder_output = embeddings

for _ in range(num_layers):
    encoder_output = TransformerEncoderLayer(
        embed_dim,
        dense_dim,
        num_heads
    )(encoder_output)

mlm_output = Dense(
    vocab_size,
    activation="softmax"
)(encoder_output)

bert = Model(
    encoder_input,
    mlm_output
)

## Compile Model

In [7]:
bert.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

bert.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 20)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ positional_embedding            │ (None, 20, 128)        │        66,560 │
│ (PositionalEmbedding)           │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder_layer       │ (None, 20, 128)        │       330,240 │
│ (TransformerEncoderLayer)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder_layer_1     │ (None, 20, 128)        │       330,240 │
│ (TransformerEncoderLayer)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 20, 500)        │        64,500 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 791,540 (3.02 MB)

 Trainable params: 791,540 (3.02 MB)

 Non-trainable params: 0 (0.00 B)

## Train Model

In [8]:
history = bert.fit(
    masked_inputs,
    mlm_labels,
    epochs=30,
    batch_size=2,
    verbose=1
)

Epoch 1/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 25s 88ms/step - accuracy: 0.4850 - loss: 4.0560 
Epoch 2/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.6100 - loss: 2.5723
Epoch 3/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.6400 - loss: 1.9810
Epoch 4/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.7250 - loss: 1.5102
Epoch 5/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 103ms/step - accuracy: 0.8200 - loss: 1.1705
Epoch 6/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.9400 - loss: 0.8837
Epoch 7/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 84ms/step - accuracy: 0.9450 - loss: 0.6548
Epoch 8/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.9750 - loss: 0.4715
Epoch 9/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 100ms/step - accuracy: 0.9750 - loss: 0.3472
Epoch 10/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 90ms/step - accuracy: 0.9750 - loss: 0.2549
Epoch 11/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 139ms/step - accuracy: 0.9900 - loss: 0.1936
Epoch 12/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 92ms/step - accuracy: 1.0000 - loss: 0.1

## Final Metrics

In [9]:
final_acc = history.history['accuracy'][-1]
final_loss = history.history['loss'][-1]

print("\nFinal Accuracy:", round(final_acc * 100, 2), "%")
print("Final Loss:", round(final_loss, 4))


Final Accuracy: 100.0 %
Final Loss: 0.0158


## Inference - Predict Masked Tokens

In [10]:
sample_text = "the quick brown fox jumps over the lazy dog"
sample_tokens = text_vectorization([sample_text])

# Mask some tokens
sample_input = sample_tokens.numpy().copy()
mask_indices = [2, 4, 6]
mask_token_id = vocab_size - 1

for idx in mask_indices:
    sample_input[0, idx] = mask_token_id

prediction = bert.predict(
    tf.constant(sample_input, dtype=tf.int64)
)

print("\nPredicted tokens for masked positions:")
vocab = text_vectorization.get_vocabulary()

for mask_idx in mask_indices:
    predicted_id = tf.argmax(prediction[0, mask_idx]).numpy()
    if predicted_id < len(vocab):
        print(f"Position {mask_idx}: {vocab[predicted_id]}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step

Predicted tokens for masked positions:
Position 2: all
Position 4: improves
Position 6: yields
